# 02 — Data preparation, integration and databases

This notebook makes all integration decisions auditable: matching keys, duplicate handling, plausibility filters, weather join coverage and row changes.

In [ ]:
import pandas as pd
from IPython.display import display
from sptdelays.settings import PATHS
from sptdelays.prepare import prepare_transport, integrate_weather
from sptdelays.collect_weather import collect_weather
from sptdelays.database import build_sqlite

## Prepare transport observations
Use `actuals` for final results. Use `live` only for development or an explicitly designed repeated-snapshot study. Delay is clipped at zero for the main outcome, while signed delay remains available for diagnostics.

In [ ]:
SOURCE = 'live'  # change to 'actuals' for the final analysis
transport = prepare_transport(SOURCE)
display(transport.head())
display(pd.read_csv(PATHS.tables / 'preparation_audit.csv'))

## Collect and integrate hourly weather
Weather is requested at each station coordinate and joined on normalized station ID plus Europe/Zurich local hour. The left join must preserve every transport row.

In [ ]:
weather = collect_weather()
model_data = integrate_weather()
display(pd.read_csv(PATHS.tables / 'integration_audit.csv'))

## SQLite and SQL from Python
The generated database contains normalized station, weather and observation tables, indexes and documented aggregation queries. Query outputs are exported to `reports/tables/`.

In [ ]:
database_path = build_sqlite()
database_path

In [ ]:
display(pd.read_csv(PATHS.tables / 'sqlite_delay_by_region_mode.csv').head(20))
display(pd.read_csv(PATHS.tables / 'sqlite_weather_delay_join.csv').head(20))

## Final validation
Explain every lost row, report the weather match rate, inspect unmatched records by region/mode and verify that the final dataset covers the intended dates. Validated DuckDB evidence is produced with `sptdelays duckdb`; the full PostgreSQL option uses `sptdelays postgres`.